# Black Summer 2019–20 — Probability Ratio (nonstationary GEV shift-fit)

Replaces the earlier "detrended ERA5 + CMIP6 hist-nat" approach. The counterfactual
is now built from the **observed ERA5 record itself**, rescaled to a pre-industrial
climate using a GMST covariate (WWA-style shift fit), with a **GEV** tail fitted to
the seasonal block maxima.

- **P1 (factual)**: each season's fire-season (Oct–Mar) max of monthly-mean daily-max
  temperature, rescaled to the 2019 climate.
- **P0 (counterfactual)**: the same pool rescaled to the pre-industrial covariate (GMST = 0
  vs 1850–1900). The shift coefficient is the regional warming response β (°C local per °C global).
- **Primary β = 0.726** (ERA5-observed fire-season amplification, notebook 05). Sensitivities:
  CMIP6 annual-tas amplification (0.935) and a data-driven fitted β.

No CMIP6 streaming is required for the PR — the method is fully reproducible from local
ERA5 + the FaIR GMST parquet. CMIP6 hist-nat (notebook 03) remains a documented null-result
cross-check only.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('../../').resolve()
sys.path.insert(0, str(ROOT))
from src.attribution import (
    area_weighted_series, season_block_max, wet_season_max_ndays,
    load_gmst, extrapolate_to, smoothed_covariate, event_gmst_sigma,
    shift_fit_gev, fit_gev, build_liability_table, far,
    AUD_TO_USD, CC_RATE_STANDARD, CC_RATE_HIGH, CLIM_START, CLIM_END,
)
from scipy.stats import genextreme

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
FIGS = ROOT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
print('Setup complete.')


In [ ]:
# ── Region / season ──
LAT_S, LAT_N = -44, -28
LON_W, LON_E = 138, 154
FIRE_MONTHS  = [10, 11, 12, 1, 2, 3]
EVENT_YEAR   = 2019
ERA5_PATH = RAW / 'era5' / 'era5_mx2t_daily_se_australia_1961_2020.nc'

ds = xr.open_dataset(ERA5_PATH)
var = next(v for v in ['mx2t', 'maximum_2m_temperature'] if v in ds)       if any(v in ds for v in ['mx2t', 'maximum_2m_temperature']) else list(ds.data_vars)[0]
ts_daily = area_weighted_series(ds[var])
if ts_daily.mean() > 100:
    ts_daily = ts_daily - 273.15

# Fire-season block maxima of monthly-mean daily-max (matches CMIP6 tasmax definition),
# anomaly vs 1961–1990. Incomplete edge seasons are dropped inside season_block_max.
fire_season = season_block_max(ts_daily.resample('ME').mean(), FIRE_MONTHS)
anom = fire_season - fire_season.loc[CLIM_START:CLIM_END].mean()
print(f'Fire-season anomaly pool: {len(anom)} complete seasons '
      f'({anom.index.min()}–{anom.index.max()})')
print(f'{EVENT_YEAR} anomaly: {anom.loc[EVENT_YEAR]:.3f} °C  '
      f'(exceeded by {(anom > anom.loc[EVENT_YEAR]).sum()} seasons)')
print('Top 5 seasons:', {int(k): round(v, 2) for k, v in anom.nlargest(5).items()})


In [ ]:
# ── GMST covariate (FaIR p50, smoothed) and event-year uncertainty ──
gmst = load_gmst(PROC)
covariate = smoothed_covariate(gmst['t_p50'])           # anomaly vs 1850–1900
g_sigma   = event_gmst_sigma(gmst, EVENT_YEAR)
print(f'GMST({EVENT_YEAR}) smoothed = {covariate.loc[EVENT_YEAR]:.3f} °C vs pre-industrial '
      f'(±{g_sigma:.3f} 1σ from FaIR)')

# ── Regional warming response β (°C local per °C global) ──
obs_af   = pd.read_csv(PROC / 'observed_amplification_factor.csv')
BETA_OBS = float(obs_af.set_index('source').loc['ERA5_observed', 'amplification'])   # 0.726
au_af    = pd.read_csv(PROC / 'au_amplification_factor.csv')
BETA_CMIP6 = float(au_af['amplification'].median())                                  # 0.935 (annual tas)
print(f'β primary (ERA5 fire-season amplification): {BETA_OBS:.3f}')
print(f'β sensitivity (CMIP6 annual-tas amplification): {BETA_CMIP6:.3f}')


In [ ]:
# ── Shift-fit PR: primary (β=0.726) + sensitivities ──
methods = {
    'primary (β=0.726, ERA5 obs)': dict(beta=BETA_OBS),
    'sens (β=0.935, CMIP6 tas)':   dict(beta=BETA_CMIP6),
    'sens (β fitted, data-driven)': dict(beta=None),
}
results = {}
for name, kw in methods.items():
    r = shift_fit_gev(anom, covariate, EVENT_YEAR, mode='additive',
                      g_event_sigma=g_sigma, **kw)
    results[name] = r
    print(f'{name:32s}: {r.summary()}')

primary = results['primary (β=0.726, ERA5 obs)']
print(f'\nPRIMARY  PR = {primary.pr:.2f} [{primary.pr_p05:.2f}–{primary.pr_p95:.2f}]  '
      f'FAR = {primary.far:.3f}')
print('WWA validation reference: FWI PR ≥ 4 (FAR ≥ 0.75), MSR PR ≥ 9 — '
      'van Oldenborgh et al. (2021)')


In [ ]:
# ── Persist PR table + bootstrap samples ──
rows = []
for name, r in results.items():
    rows.append({'method': name, 'beta': r.beta, 'pr': r.pr,
                 'pr_p05': r.pr_p05, 'pr_p95': r.pr_p95, 'far': r.far,
                 'gev_xi': -r.gev_params[0]})
rows.append({'method': 'WWA FWI (van Oldenborgh 2021)', 'beta': np.nan, 'pr': 4.0,
             'pr_p05': 4.0, 'pr_p95': np.nan, 'far': 0.75, 'gev_xi': np.nan})
pr_df = pd.DataFrame(rows)
pr_df.to_csv(PROC / 'black_summer_pr_era5.csv', index=False)
pd.DataFrame({'pr_boot': primary.pr_boot}).to_parquet(
    PROC / 'black_summer_pr_shiftfit_bootstrap.parquet', index=False)
print('Saved black_summer_pr_era5.csv and black_summer_pr_shiftfit_bootstrap.parquet')
print(pr_df.to_string(index=False))


In [ ]:
# ── Figure: distributions + PR vs threshold ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
xi, loc, scale = primary.gev_params
x = np.linspace(anom.min() - 0.5, anom.max() + 1.0, 300)
shift = primary.beta * covariate.loc[EVENT_YEAR]

ax = axes[0]
ax.hist(anom.values, bins=16, density=True, alpha=0.35, color='#90A4AE',
        label='ERA5 seasons (raw)')
ax.plot(x, genextreme.pdf(x, xi, loc, scale), color='#FF5722', lw=2.5,
        label='P1 factual GEV (2019 climate)')
ax.plot(x, genextreme.pdf(x, xi, loc - shift, scale), color='#2196F3', lw=2.5,
        label='P0 counterfactual GEV (pre-industrial)')
ax.axvline(primary.threshold, color='k', ls='--', lw=1.5,
           label=f'2019 event ({primary.threshold:.2f}°C)')
ax.set_xlabel('Fire-season max tasmax anomaly (°C)')
ax.set_ylabel('Density')
ax.set_title('Shift-fit GEV: factual vs counterfactual', fontsize=11)
ax.legend(fontsize=8)

ax2 = axes[1]
boot = primary.pr_boot
boot = boot[np.isfinite(boot)]
ax2.hist(np.clip(boot, 0, 30), bins=40, density=True, alpha=0.7, color='#673AB7')
ax2.axvline(primary.pr, color='k', lw=2, ls='--', label=f'median PR = {primary.pr:.1f}')
ax2.axvline(4.0, color='grey', lw=1.2, ls=':', label='WWA FWI lower bound (PR=4)')
ax2.set_xlabel('Probability Ratio')
ax2.set_title('Bootstrap PR distribution (β=0.726)', fontsize=11)
ax2.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / 'black_summer_pr_shiftfit.png', bbox_inches='tight')
plt.show()
print('Saved figure.')
